# Fit a successful benchmark case with its selected skeleton

The Cesàro-equation case **eponymous_144** is

$$y=\frac{\exp(x_0/x_1)}{x_1}.$$

The completed KS-IES benchmark selected **IAPB05** (202 trainable parameters) by mean log validation NMSE over ten seeds, after all 18 candidates finished. Its recorded ten-seed test GNMSE is **2.5236e-9** (seed 421: **3.0178e-9** NMSE). `benchmark_equation.json` records the result hash, selected structure and source-derived per-map budgets. This is a retrospectively chosen illustration, not another unbiased benchmark result.

The corresponding frozen bank was built from **Physics + Biology/Chemistry + the symbolic-regression benchmark (VSR-DPG modified Livermore2)**, excluding Mathematics. This notebook reuses that bank and retains the audited input ranges and sampling rule. It does not rebuild from the target expression.

We train **one model with the project's standalone L-BFGS**, not `torch.optim.LBFGS` or seed batching. Defaults: seed 421, 10,000 fresh points per split, 50 outer calls, at most 20 inner iterations, float64. A new standalone run need not equal the archived seed-batched result. Frozen filenames retain `v2` only as the KS-IES compatibility identifier.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime, timezone
import copy
import json
import math
import sys
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "structured_kan").is_dir())
sys.path.insert(0, str(ROOT))
from examples.common import benchmark_equation, build_model, metrics, save, standardization
from structured_kan.dataset.fresh_sampling import draw_unit
from structured_kan.model.StructuredKANBuilder import StructuredKANBuilder
from structured_kan.optimizer.lbfgs import LBFGS

args = SimpleNamespace(
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=421, builder="IAPB05", points=10000, outer_steps=50,
    output=ROOT / "structured_kan/data/results" /
        ("notebook_fit_equation_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")),
)
print({"device": args.device, "torch": torch.__version__,
       "optimizer": "project standalone LBFGS", "seed_batch": False})

## Sample data and standardize

All normalization statistics come from the training split. Validation and test points are disjoint independent samples.

In [ ]:
row, reference = benchmark_equation()
if args.builder != reference['builder']:
    raise ValueError('this example uses the benchmark-selected IAPB05 and its archived per-map budgets')
bounds = torch.tensor(row['quality_audit']['support']['bounds'], dtype=torch.float64)

def sample(split):
    unit = draw_unit(row['case_id'], args.seed, split, args.points, 2)
    x = bounds[:, 0] + (bounds[:, 1]-bounds[:, 0])*torch.from_numpy(unit)
    x = x.to(args.device)
    y = torch.exp(x[:, :1]/x[:, 1:2])/x[:, 1:2]
    return x, y

x_train, y_train = sample('train')
x_val, y_val = sample('validation')
x_test, y_test = sample('test')
x_mean, x_std = standardization(x_train)
y_mean, y_std = standardization(y_train)
x_fit, y_fit = (x_train-x_mean)/x_std, (y_train-y_mean)/y_std

print({"case": row["case_id"], "bounds": bounds.tolist(), "reference": reference["reference"]})

## Materialize the corresponding held-out skeleton

IAPB05 and its per-map budgets are read from the selected benchmark configuration. The bank/source hashes and Mathematics exclusion are checked. Grids use training inputs only; source-role sharing and private affine routes are unchanged.

In [ ]:
model, spec, metadata = build_model(args, x_fit, bank_path=ROOT/reference['catalogue'],
                                  route_budgets=reference['private_phi_G_by_route'])
assert metadata['parameters'] == reference['parameters']

print({k: metadata[k] for k in ["constructor", "builder", "parameters", "bank_sha256"]})

## Standalone optimizer and closure

Each closure recomputes the complete loss and calls `backward()`. There is one model and one L-BFGS history. Strong-Wolfe trials use this same closure.

In [ ]:
optimizer = LBFGS(
    model.parameters(), lr=1., max_iter=20, history_size=100,
    tolerance_grad=1e-32, tolerance_change=1e-32, tolerance_ys=1e-32,
    line_search_fn="strong_wolfe", two_loop_mode="nosync_fma",
    scalar_mode="coalesced_host",
)

def predict(x):
    return model((x-x_mean)/x_std)*y_std + y_mean

# An ordinary single-model closure: zero_grad, forward, backward, loss.
def closure():
    optimizer.zero_grad(set_to_none=True)
    loss = (model(x_fit)-y_fit).square().mean()
    loss.backward()
    return loss

@torch.no_grad()
def validation_loss():
    return metrics(predict(x_val), y_val)['nmse']

## Train and retain the best validation checkpoint

Validation selects a checkpoint of this single skeleton; test observations remain report-only.

In [ ]:
best_loss = float(validation_loss())
best_state = copy.deepcopy(model.state_dict())
best_step = 0
for step in range(1, args.outer_steps + 1):
    optimizer.step(closure)
    loss = float(validation_loss())
    if not math.isfinite(loss):
        raise FloatingPointError("non-finite validation error")
    if loss < best_loss:
        best_loss, best_state, best_step = loss, copy.deepcopy(model.state_dict()), step
    if step == 1 or step % 10 == 0 or step == args.outer_steps:
        print(f"outer={step:2d}, validation NMSE={loss:.6e}")
model.load_state_dict(best_state)
print({"selected_outer_step": best_step})

## Report original-scale MSE and normalized MSE; save the model

In [ ]:
with torch.no_grad():
    result = dict(task='equation', case_id=row['case_id'], equation=row['expression'],
                  support=bounds.tolist(), benchmark_reference=reference['reference'],
                  points_per_split=args.points, selected_outer_step=best_step,
                  train=metrics(predict(x_train), y_train),
                  validation=metrics(predict(x_val), y_val),
                  test=metrics(predict(x_test), y_test))
save(args, model, spec, metadata,
     dict(x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std), result)


## Verify the saved checkpoint

In [ ]:
checkpoint = torch.load(args.output / "model.pt", map_location=args.device, weights_only=True)
restored = StructuredKANBuilder(2, dtype=torch.float64, device=args.device).build(checkpoint["spec"])
restored.load_state_dict(checkpoint["state_dict"])
stats = checkpoint["normalizers"]
with torch.no_grad():
    check_x = x_test
    restored_prediction = restored((check_x-stats["x_mean"])/stats["x_std"])*stats["y_std"] + stats["y_mean"]
    torch.testing.assert_close(restored_prediction, predict(check_x), rtol=0., atol=0.)
print("Saved checkpoint reproduces predictions exactly.")

## Visualize held-out predictions

In [ ]:
import matplotlib.pyplot as plt
with torch.no_grad():
    prediction = predict(x_test).cpu().numpy().ravel()
    target = y_test.cpu().numpy().ravel()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(target, prediction, s=7, alpha=.5)
lo, hi = float(target.min()), float(target.max())
axes[0].plot([lo, hi], [lo, hi], color="black", linewidth=1)
axes[0].set(xlabel="Exact value", ylabel="Prediction", title="Held-out equation fit")
axes[1].scatter(target, prediction-target, s=7, alpha=.5)
axes[1].set(xlabel="Exact value", ylabel="Prediction minus target", title="Held-out residual")
fig.tight_layout()
plt.show()